"""
LAB 5: MapReduce -- Fault Tolerance & Stragglers (maps to Lecture 5, slide 7)
--------------------------------------------------------------------------------
Builds on Lab 4. Same word-count job, but now bad things happen to
some of the map tasks, and the MASTER has to cope -- exactly like the
slide says: "Across a cluster of thousands of machines, failures are
the norm... MapReduce's runtime -- not the programmer -- absorbs them."

Two failure types simulated here:

  1. WORKER CRASH   -> task raises an exception (worker "died").
                       Master notices and RE-EXECUTES the task.
  2. STRAGGLER       -> task takes way too long (slow machine).
                       Master launches a BACKUP ("speculative")
                       copy of the same task on another worker and
                       uses whichever copy finishes first.

Run this a few times -- the crash/straggler doc is picked on purpose
here (doc 1 crashes once, doc 3 is always slow), so you can watch the
recovery logic every run.
"""

In [1]:
import multiprocessing as mp

In [2]:
import os

In [3]:
import time

In [4]:
DOCUMENTS = [
    "distributed systems are fun",
    "map reduce is a programming model",
    "the master splits input into map tasks",
    "reduce tasks sum the counts from map tasks",
    "fun fun fun distributed computing",
]

In [5]:
MAP_TIMEOUT = 2.0  # seconds the master is willing to wait for one map task

In [6]:
# A map task that sometimes "misbehaves" on purpose, so we can
# watch the master's recovery logic kick in.
#   doc_id 1 -> always crashes on its FIRST attempt (simulates a
#               worker machine dying mid-task).
#   doc_id 3 -> always runs slowly (simulates a straggler machine).
def flaky_map_task(doc_id, text, attempt):
    pid = os.getpid()
    print(f"[MAP worker pid={pid}] attempt #{attempt} on doc {doc_id}: {text!r}")

    if doc_id == 1 and attempt == 1:
        print(f"[MAP worker pid={pid}] !! CRASHING on doc {doc_id} (simulated failure)")
        raise RuntimeError("simulated worker crash")

    if doc_id == 3:
        time.sleep(3.0)  # deliberately slower than MAP_TIMEOUT -> straggler

    pairs = [(word, 1) for word in text.split()]
    return pairs

In [7]:
# MASTER logic: run one map task, and RE-EXECUTE it (possibly many
# times) if the worker crashed. This is the "re-execution" bullet
# from slide 7.
def run_map_with_retry(pool, doc_id, text, max_attempts=3):
    for attempt in range(1, max_attempts + 1):
        async_result = pool.apply_async(flaky_map_task, (doc_id, text, attempt))
        try:
            return async_result.get(timeout=MAP_TIMEOUT)
        except mp.TimeoutError:
            print(f"[MASTER] doc {doc_id} attempt #{attempt} is a STRAGGLER "
                  f"(> {MAP_TIMEOUT}s) -- launching a BACKUP copy")
            # Speculative execution: race a second copy in parallel.
            backup_result = pool.apply_async(flaky_map_task, (doc_id, text, attempt))
            # Whichever of the two finishes first, we use. For simplicity
            # here we just wait (without a timeout) on the backup, since
            # it's the one we expect to win the race in this demo.
            return backup_result.get()
        except Exception as e:
            print(f"[MASTER] doc {doc_id} attempt #{attempt} CRASHED ({e}); "
                  f"re-executing on a fresh worker")
            continue
    raise RuntimeError(f"doc {doc_id} failed after {max_attempts} attempts")

In [8]:
def main():
    print("[MASTER] Splitting input into", len(DOCUMENTS), "map tasks")

    with mp.Pool(processes=4) as pool:
        all_pairs = []
        for doc_id, text in enumerate(DOCUMENTS):
            pairs = run_map_with_retry(pool, doc_id, text)
            all_pairs.extend(pairs)

    # Simple reduce step (sequential here, since the focus of this
    # lab is fault tolerance in the MAP phase).
    counts = {}
    for word, c in all_pairs:
        counts[word] = counts.get(word, 0) + c

    print("\n[MASTER] All map tasks eventually succeeded. FINAL WORD COUNTS:")
    for word, count in sorted(counts.items(), key=lambda kv: -kv[1]):
        print(f"  {word:12s} {count}")

In [9]:
if __name__ == "__main__":
    main()

[MASTER] Splitting input into 5 map tasks


[MAP worker pid=210] attempt #1 on doc 0: 'distributed systems are fun'

[MAP worker pid=211] attempt #1 on doc 1: 'map reduce is a programming model'

[MAP worker pid=211] !! CRASHING on doc 1 (simulated failure)

[MAP worker pid=212] attempt #2 on doc 1: 'map reduce is a programming model'

...

[MASTER] doc 1 attempt #1 CRASHED (simulated worker crash); re-executing on a fresh worker


[MAP worker pid=211] attempt #1 on doc 3: 'reduce tasks sum the counts from map tasks'

[MASTER] doc 3 attempt #1 is a STRAGGLER (> 2.0s) -- launching a BACKUP copy


[MAP worker pid=212] attempt #1 on doc 4: 'fun fun fun distributed computing'


[MASTER] All map tasks eventually succeeded. FINAL WORD COUNTS:
  fun          4
  map          3
  tasks        3
  distributed  2
  reduce       2
  the          2
  systems      1
  are          1
  is           1
  a            1
  programming  1
  model        1
  master       1
  splits       1
  input        1
  into         1
  sum          1
  counts       1
  from         1
  computing    1
